In [24]:
import pandas as pd

In [25]:
df = pd.read_csv('./datos/Base Haciendas Depurada.csv', sep=';', decimal=',')

In [26]:
df.head()

,FECHA,Semana,Zona,Unidad,Nombre_Unidad,Real,Costo_Ha,Atencion_Plantacion,C_Riego,C_Mano_Obra_Riego,...,Precipitacion_mm,Evotranspiracion,Humedad,Ausentismo_Agricola,Ausentismo_Justificado_Agricola,Ausentismo_Injustificado_Agricola,RotPerson_Salida_Todos_Motivos_Agricola,Pago_Labor_Persona,Pago_Por_Cuenta,Vacante_Labor
0,1/01/20,5,Zona Camarones,2283,Hacienda San Escriva,5.114322,926.408722,120674.0,1576,643,...,84.40,14.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0
1,1/02/20,9,Zona Camarones,2283,Hacienda San Escriva,6.033120,1010.523197,131428.0,2253,540,...,112.25,17.00,0.0,0.0,0.0,0.0,0.006803,0.0,0.0,0
2,1/03/20,14,Zona Camarones,2283,Hacienda San Escriva,4.495248,831.462938,107817.0,778,515,...,101.75,16.25,0.0,0.0,0.0,0.0,0.013423,0.0,0.0,0
3,1/04/20,18,Zona Camarones,2283,Hacienda San Escriva,6.130292,990.703324,125872.0,1786,720,...,103.60,13.40,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0
4,1/05/20,22,Zona Camarones,2283,Hacienda San Escriva,6.329382,1028.251467,135611.0,3507,591,...,50.50,14.50,0.0,0.0,0.0,0.0,0.006211,0.0,0.0,0


In [27]:
# Instalar langchain con soporte para Gemini
import subprocess
import sys
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

try:
    import langchain
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain"])

try:
    import langchain_core
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-core"])

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-google-genai"])
    from langchain_google_genai import ChatGoogleGenerativeAI
    
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [32]:
# Crear el agente con LangChain - Google Gemini
# GEMINI_API_KEY y GEMINI_MODEL se cargan desde el archivo .env usando load_dotenv

# Crear el prompt para el agente
prompt_template = PromptTemplate(
    input_variables=["question"],
    template="""Eres un experto en análisis de datos con pandas. 
Tienes acceso a un DataFrame llamado 'df' con datos sobre haciendas.

Basándote en la siguiente pregunta, genera SOLO código Python que resuelva la pregunta.
El código debe:
1. Usar el DataFrame 'df' que ya está disponible
2. Retornar un resultado (print, o guardar en una variable)
3. Ser ejecutable directamente con exec()

Pregunta: {question}

Retorna SOLO el código Python, sin explicaciones adicionales, sin markdown, sin triple backticks."""
)

# Configurar Google Gemini usando variables del .env
gemini_key = os.getenv("GEMINI_API_KEY")
if not gemini_key:
    raise ValueError("⚠️  GEMINI_API_KEY no configurada en .env")

gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")  # valor por defecto si no está en .env

llm = ChatGoogleGenerativeAI(
    model=gemini_model,
    temperature=0,
    google_api_key=gemini_key
)
print(f"✅ Usando Google {gemini_model}")

# Crear la cadena LLM con el operador pipe (|)
output_parser = StrOutputParser()
agent_chain = prompt_template | llm | output_parser

def agent_generate_code(question):
    """
    Agente que recibe una pregunta y retorna código Python que la resuelve.
    
    Args:
        question (str): La pregunta sobre los datos
        
    Returns:
        str: Código Python que resuelve la pregunta
    """
    code = agent_chain.invoke({"question": question})
    return code.strip()

✅ Usando Google gemini-2.0-flash


In [29]:
# Función para ejecutar código con acceso al DataFrame
def execute_code(code_string, df_context):
    """
    Ejecuta código Python generado con acceso seguro al DataFrame.
    
    Args:
        code_string (str): Código Python a ejecutar
        df_context (pd.DataFrame): El DataFrame a pasar como contexto
        
    Returns:
        str or object: El resultado de la ejecución
    """
    # Crear un diccionario local con el DataFrame y librerías útiles
    local_vars = {
        'df': df_context,
        'pd': pd,
        'result': None
    }
    
    # Permitir código peligroso (necesario para exec)
    exec(code_string, {"__builtins__": __builtins__, "pd": pd, "print": print}, local_vars)
    
    return local_vars.get('result', "Código ejecutado sin retorno explícito")


def query_and_execute(question, df_context):
    """
    Función principal que integra el agente con la ejecución del código.
    
    Args:
        question (str): Pregunta sobre los datos
        df_context (pd.DataFrame): El DataFrame a analizar
        
    Returns:
        tuple: (código_generado, resultado_ejecución)
    """
    print(f"💡 Pregunta: {question}\n")
    
    # Generar el código con el agente
    print("🤖 Generando código con LangChain...")
    generated_code = agent_generate_code(question)
    
    print(f"📝 Código generado:\n{generated_code}\n")
    
    # Ejecutar el código
    print("▶️  Ejecutando código...")
    try:
        result = execute_code(generated_code, df_context)
        print(f"✅ Resultado:\n{result}")
        return generated_code, result
    except Exception as e:
        print(f"❌ Error al ejecutar: {str(e)}")
        return generated_code, f"Error: {str(e)}"

In [30]:
# Ejemplo de uso
# IMPORTANTE: Configura una de estas variables de entorno:
# - GOOGLE_API_KEY para usar Gemini (recomendado - más económico)
# - ANTHROPIC_API_KEY para usar Claude

if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 DEMO: Agente de Análisis de Datos")
    print("="*60 + "\n")
    
    # Ejemplo 1: Pregunta simple
    question1 = "¿Cuántas filas tiene el DataFrame?"
    code1, result1 = query_and_execute(question1, df)
    
    print("\n" + "="*60 + "\n")
    
    # Ejemplo 2: Análisis más complejo
    question2 = "¿Cuál es el costo promedio (Costo_Ha) por zona?"
    code2, result2 = query_and_execute(question2, df)


🚀 DEMO: Agente de Análisis de Datos

💡 Pregunta: ¿Cuántas filas tiene el DataFrame?

🤖 Generando código con LangChain...


ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
# CÓMO USAR:
# 1. El archivo .env debe contener: GEMINI_API_KEY="tu-clave-api"
# 2. load_dotenv() carga automáticamente las variables
# 3. ChatGoogleGenerativeAI usa gemini-1.5-flash (modelo económico)

print("✅ Agente listo para usar")